
# EMT DC SSN Pi-Line Source Step — Direct Pybind Simulation

This notebook creates and runs the complete circuit through `dpsimpy`.

```text
20 kV → 22 kV ideal DC source step at 0.1 s
        │
      0.2 Ω feeder
        │
    sending node
        │
DC SSN Pi-line: R = 0.5 Ω, L = 20 mH, total C = 2 mF
        │
   receiving node
        │
      20 Ω load
        │
      ground
```

Topology construction, parameterization, EMT stepping, source disturbance,
logging, result loading, and plotting are all controlled from Python. The
underlying numerical solver remains the compiled DPsim C++ core accessed through
pybind.


In [ ]:
from __future__ import annotations

import re
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

## 1. Import the locally built `dpsimpy`

In [ ]:
REPO_ROOT = Path.home() / "dpsim"
BUILD_DIR = REPO_ROOT / "build"

assert REPO_ROOT.is_dir(), f"Repository not found: {REPO_ROOT}"
assert BUILD_DIR.is_dir(), f"Build directory not found: {BUILD_DIR}"

module_candidates = list(BUILD_DIR.rglob("dpsimpy*.so"))
if not module_candidates:
    module_candidates = list(BUILD_DIR.rglob("dpsimpy*.pyd"))

if module_candidates:
    module_file = max(module_candidates, key=lambda path: path.stat().st_mtime)
    sys.path.insert(0, str(module_file.parent))
    print(f"Using local module: {module_file}")
else:
    print("No local dpsimpy binary found below build/; using installed module.")

In [ ]:
import dpsimpy

print(f"dpsimpy version: {getattr(dpsimpy, '__version__', 'unknown')}")
print(f"dpsimpy module:  {dpsimpy.__file__}")

## 2. Verify the scalar DC bindings

In [ ]:
checks = {
    "PhaseType.DC": hasattr(dpsimpy.PhaseType, "DC"),
    "emt.dc": hasattr(dpsimpy.emt, "dc"),
}

if checks["emt.dc"]:
    checks.update(
        {
            "emt.dc.VoltageSource": hasattr(dpsimpy.emt.dc, "VoltageSource"),
            "emt.dc.CurrentSource": hasattr(dpsimpy.emt.dc, "CurrentSource"),
            "emt.dc.ssn": hasattr(dpsimpy.emt.dc, "ssn"),
        }
    )

if checks.get("emt.dc.ssn", False):
    for name in ("Resistor", "Capacitor", "Inductor", "PiLine"):
        checks[f"emt.dc.ssn.{name}"] = hasattr(dpsimpy.emt.dc.ssn, name)

display(pd.Series(checks, name="available").to_frame())

missing = [name for name, available in checks.items() if not available]
if missing:
    raise RuntimeError(
        "Missing scalar DC bindings:\n  - "
        + "\n  - ".join(missing)
        + "\n\nApply apply_emt_dc_pybind.py and rebuild dpsimpy."
    )

## 3. Parameters and analytical operating points

In [ ]:
SIM_NAME = "EMT_DC_SSN_PiLine_SourceStep_Pybind"

TIME_STEP = 1e-5
FINAL_TIME = 0.5
SOURCE_STEP_TIME = 0.1

INITIAL_SOURCE_VOLTAGE = 20e3
STEPPED_SOURCE_VOLTAGE = 22e3

FEEDER_RESISTANCE = 0.2
LINE_RESISTANCE = 0.5
LINE_INDUCTANCE = 20e-3
TOTAL_LINE_CAPACITANCE = 2e-3
TOTAL_LINE_CONDUCTANCE = 0.0
LOAD_RESISTANCE = 20.0

NUMBER_OF_STEPS = int(round(FINAL_TIME / TIME_STEP))
SOURCE_STEP_INDEX = int(round(SOURCE_STEP_TIME / TIME_STEP))

In [ ]:
def dc_operating_point(source_voltage: float) -> dict[str, float]:
    current = source_voltage / (FEEDER_RESISTANCE + LINE_RESISTANCE + LOAD_RESISTANCE)
    return {
        "source_voltage": source_voltage,
        "current": current,
        "sending_voltage": source_voltage - FEEDER_RESISTANCE * current,
        "receiving_voltage": LOAD_RESISTANCE * current,
        "line_voltage_drop": LINE_RESISTANCE * current,
    }


initial_expected = dc_operating_point(INITIAL_SOURCE_VOLTAGE)
final_expected = dc_operating_point(STEPPED_SOURCE_VOLTAGE)

pd.DataFrame(
    [initial_expected, final_expected],
    index=["pre-step", "post-step"],
)

## 4. Construct the DC topology through pybind

In [ ]:
source_node = dpsimpy.emt.SimNode("source_node", dpsimpy.PhaseType.DC)
sending_node = dpsimpy.emt.SimNode("sending_node", dpsimpy.PhaseType.DC)
receiving_node = dpsimpy.emt.SimNode("receiving_node", dpsimpy.PhaseType.DC)

source_node.set_initial_voltage(complex(INITIAL_SOURCE_VOLTAGE, 0.0))
sending_node.set_initial_voltage(complex(initial_expected["sending_voltage"], 0.0))
receiving_node.set_initial_voltage(complex(initial_expected["receiving_voltage"], 0.0))

gnd = dpsimpy.emt.SimNode.gnd

In [ ]:
source = dpsimpy.emt.dc.VoltageSource("dc_source")
source.set_parameters(INITIAL_SOURCE_VOLTAGE)
source.connect([gnd, source_node])

feeder = dpsimpy.emt.dc.ssn.Resistor("feeder")
feeder.set_parameters(FEEDER_RESISTANCE)
feeder.connect([sending_node, source_node])

line = dpsimpy.emt.dc.ssn.PiLine("dc_pi_line")
line.set_parameters(
    LINE_RESISTANCE,
    LINE_INDUCTANCE,
    TOTAL_LINE_CAPACITANCE,
    TOTAL_LINE_CONDUCTANCE,
    initial_expected["current"],
)
line.connect([receiving_node, sending_node])

load = dpsimpy.emt.dc.ssn.Resistor("dc_load")
load.set_parameters(LOAD_RESISTANCE)
load.connect([gnd, receiving_node])

nodes = [source_node, sending_node, receiving_node]
components = [source, feeder, line, load]

system = dpsimpy.SystemTopology(0.0, nodes, components)

print("Topology created through dpsimpy.")

## 5. Configure the DPsim logger

In [ ]:
log_dir = REPO_ROOT / "logs" / SIM_NAME

if log_dir.exists():
    shutil.rmtree(log_dir)
log_dir.mkdir(parents=True, exist_ok=True)

dpsimpy.Logger.set_log_dir(str(log_dir))
logger = dpsimpy.Logger(SIM_NAME)

logger.log_attribute("v_source_node", "v", source_node)
logger.log_attribute("v_sending_node", "v", sending_node)
logger.log_attribute("v_receiving_node", "v", receiving_node)

logger.log_attribute("v_source_intf", "v_intf", source)
logger.log_attribute("i_source_intf", "i_intf", source)

logger.log_attribute("v_feeder_intf", "v_intf", feeder)
logger.log_attribute("i_feeder_intf", "i_intf", feeder)

logger.log_attribute("v_line_intf", "v_intf", line)
logger.log_attribute("i_line_intf", "i_intf", line)

logger.log_attribute("v_load_intf", "v_intf", load)
logger.log_attribute("i_load_intf", "i_intf", load)

print(f"Logging to: {log_dir}")

## 6. Run the EMT simulation from Python

In [ ]:
simulation = dpsimpy.Simulation(SIM_NAME, dpsimpy.LogLevel.info)
simulation.set_system(system)
simulation.set_domain(dpsimpy.Domain.EMT)
simulation.set_solver(dpsimpy.Solver.MNA)
simulation.set_time_step(TIME_STEP)
simulation.set_final_time(FINAL_TIME)
simulation.add_logger(logger)

simulation.start()

source_changed = False

for step in range(NUMBER_OF_STEPS):
    if not source_changed and step >= SOURCE_STEP_INDEX:
        source.set_parameters(STEPPED_SOURCE_VOLTAGE)
        source_changed = True
        print(
            f"Source step at t = {step * TIME_STEP:.6f} s: "
            f"{INITIAL_SOURCE_VOLTAGE:.1f} V -> "
            f"{STEPPED_SOURCE_VOLTAGE:.1f} V"
        )

    simulation.next()

simulation.stop()

if not source_changed:
    raise RuntimeError("Source step was never applied.")

print("Simulation completed.")

## 7. Read the generated time series

In [ ]:
csv_candidates = list(log_dir.rglob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV generated below {log_dir}")

csv_path = max(csv_candidates, key=lambda path: path.stat().st_mtime)
results = pd.read_csv(csv_path, sep=None, engine="python")

print(f"CSV:     {csv_path}")
print(f"Rows:    {len(results)}")
print(f"Columns: {len(results.columns)}")
display(results.head())

In [ ]:
print("Logged columns:")
for column in results.columns:
    print(f"  {column}")

## 8. Resolve scalar logger columns

In [ ]:
def normalize(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def find_column(base_name: str) -> str:
    base = normalize(base_name)
    exact = [column for column in results.columns if normalize(column) == base]
    if exact:
        return exact[0]

    matches = [column for column in results.columns if base in normalize(column)]
    if not matches:
        raise KeyError(f"No column matching {base_name!r}.")

    return min(matches, key=len)


time_column = next(
    (
        column
        for column in results.columns
        if normalize(column) in {"time", "t"} or "time" in normalize(column)
    ),
    None,
)
if time_column is None:
    raise KeyError("No simulation time column found.")

columns = {
    "v_source": find_column("v_source_node"),
    "v_sending": find_column("v_sending_node"),
    "v_receiving": find_column("v_receiving_node"),
    "i_source": find_column("i_source_intf"),
    "i_feeder": find_column("i_feeder_intf"),
    "i_line": find_column("i_line_intf"),
    "i_load": find_column("i_load_intf"),
}

display(pd.Series(columns, name="CSV column").to_frame())

## 9. Validate numerical output

In [ ]:
numeric = results.select_dtypes(include=[np.number])
if numeric.empty:
    raise RuntimeError("Logger output contains no numeric columns.")
if not np.isfinite(numeric.to_numpy()).all():
    raise RuntimeError("Logger output contains NaN or Inf.")

time = results[time_column].to_numpy(dtype=float)

print(f"Time range: {time.min():.6f} s to {time.max():.6f} s")
print("All numeric logger values are finite.")

## 10. Plot node voltages

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(time, results[columns["v_source"]], label="Source node")
ax.plot(time, results[columns["v_sending"]], label="Sending node")
ax.plot(time, results[columns["v_receiving"]], label="Receiving node")
ax.axvline(SOURCE_STEP_TIME, linestyle="--", label="Source step")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Voltage [V]")
ax.set_title("Scalar DC node voltages")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 11. Plot branch currents

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(time, results[columns["i_source"]], label="Source")
ax.plot(time, results[columns["i_feeder"]], label="Feeder")
ax.plot(time, results[columns["i_line"]], label="Pi-line")
ax.plot(time, results[columns["i_load"]], label="Load")
ax.axvline(SOURCE_STEP_TIME, linestyle="--", label="Source step")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Current [A]")
ax.set_title("Scalar DC branch currents")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 12. Zoom into the transient

In [ ]:
zoom = (time >= SOURCE_STEP_TIME - 0.01) & (time <= SOURCE_STEP_TIME + 0.12)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    time[zoom],
    results.loc[zoom, columns["v_sending"]],
    label="Sending node",
)
ax.plot(
    time[zoom],
    results.loc[zoom, columns["v_receiving"]],
    label="Receiving node",
)
ax.axvline(SOURCE_STEP_TIME, linestyle="--", label="Source step")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Voltage [V]")
ax.set_title("Pi-line voltage transient")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    time[zoom],
    results.loc[zoom, columns["i_line"]],
    label="Pi-line series current",
)
ax.plot(
    time[zoom],
    results.loc[zoom, columns["i_load"]],
    label="Load current",
)
ax.axvline(SOURCE_STEP_TIME, linestyle="--", label="Source step")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Current [A]")
ax.set_title("Pi-line current transient")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()

## 13. Compare analytical and simulated steady states

In [ ]:
def mean_window(column: str, start: float, end: float) -> float:
    mask = (time >= start) & (time <= end)
    if not mask.any():
        raise ValueError(f"No samples between {start} and {end} s.")
    return float(results.loc[mask, column].mean())


pre_window = (SOURCE_STEP_TIME - 0.02, SOURCE_STEP_TIME - 0.002)
post_window = (FINAL_TIME - 0.02, FINAL_TIME)

comparison = pd.DataFrame(
    {
        "analytical_pre": {
            "sending_voltage": initial_expected["sending_voltage"],
            "receiving_voltage": initial_expected["receiving_voltage"],
            "line_current": initial_expected["current"],
        },
        "simulated_pre": {
            "sending_voltage": mean_window(columns["v_sending"], *pre_window),
            "receiving_voltage": mean_window(columns["v_receiving"], *pre_window),
            "line_current": mean_window(columns["i_line"], *pre_window),
        },
        "analytical_post": {
            "sending_voltage": final_expected["sending_voltage"],
            "receiving_voltage": final_expected["receiving_voltage"],
            "line_current": final_expected["current"],
        },
        "simulated_post": {
            "sending_voltage": mean_window(columns["v_sending"], *post_window),
            "receiving_voltage": mean_window(columns["v_receiving"], *post_window),
            "line_current": mean_window(columns["i_line"], *post_window),
        },
    }
)

comparison["pre_relative_error"] = (
    comparison["simulated_pre"] - comparison["analytical_pre"]
).abs() / comparison["analytical_pre"].abs().clip(lower=1e-12)

comparison["post_relative_error"] = (
    comparison["simulated_post"] - comparison["analytical_post"]
).abs() / comparison["analytical_post"].abs().clip(lower=1e-12)

comparison

## 14. Reconstruct powers

In [ ]:
source_power = results[columns["v_source"]] * results[columns["i_source"]]
load_power = results[columns["v_receiving"]] * results[columns["i_load"]]

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(time, source_power, label="Source interface power")
ax.plot(time, load_power, label="Load power")
ax.axvline(SOURCE_STEP_TIME, linestyle="--", label="Source step")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Power [W]")
ax.set_title("DC source and load powers")
ax.grid(True)
ax.legend()
fig.tight_layout()
plt.show()